# Principal Component Analysis — Detailed
### Dimensionality Reduction on the Breast Cancer Dataset

## 1. Project Overview
This notebook provides a comprehensive walkthrough of Principal Component Analysis (PCA) using scikit-learn's breast cancer dataset. We demonstrate how PCA reduces high-dimensional data while preserving variance, and visualise the results in 2D and 3D.

## 2. Learning Objectives
- Understand the mathematical intuition behind PCA
- Apply StandardScaler before PCA (critical preprocessing step)
- Determine the optimal number of components using explained variance
- Visualise high-dimensional data in 2D using PCA projections
- Interpret principal component loadings

## 3. Business / Research Problem
**Question:** Can we reduce the 30 features of the breast cancer dataset to 2–3 principal components while retaining most of the variance? How well do the projected components separate malignant from benign tumours?

## 4. Why This Analysis Matters
Dimensionality reduction is essential when dealing with high-dimensional datasets. PCA helps with visualisation, noise reduction, multicollinearity mitigation, and computational efficiency in downstream models.

## 5. Dataset Overview
The breast cancer dataset from scikit-learn contains:
- **569 samples**, **30 numeric features** (mean, SE, and worst of 10 measurements)
- **Target:** 0 = malignant, 1 = benign
- Features include: radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension

## 6. Dataset Source and License Notes
- **Source:** `sklearn.datasets.load_breast_cancer()`
- **Origin:** UCI Machine Learning Repository
- **License:** Public domain / CC BY 4.0

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['pandas','numpy','matplotlib','seaborn','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
RANDOM_STATE = 42
TARGET_NAMES = {0: 'Malignant', 1: 'Benign'}

## 10. Dataset Loading

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['diagnosis'] = df['target'].map(TARGET_NAMES)
print(f'Shape: {df.shape}')
print(f'Features: {len(data.feature_names)}')
print(f'Classes: {dict(zip(*np.unique(data.target, return_counts=True)))}')
df.head()

## 11. Data Validation Checks

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print(f'\nFeature ranges (sample):')
for col in data.feature_names[:5]:
    print(f'  {col}: {df[col].min():.2f} — {df[col].max():.2f}')
print(f'\nScale difference: features range from O(0.01) to O(1000+)')
print('=> StandardScaler is ESSENTIAL before PCA')

## 12. Data Cleaning
No missing values. The key preprocessing step is feature scaling.

In [ ]:
X = df[data.feature_names].values
y = df['target'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Before scaling — mean: {X.mean(axis=0)[:3].round(2)}, std: {X.std(axis=0)[:3].round(2)}')
print(f'After scaling  — mean: {X_scaled.mean(axis=0)[:3].round(4)}, std: {X_scaled.std(axis=0)[:3].round(4)}')

## 13. Exploratory Data Analysis
### Correlation structure before PCA

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
corr = pd.DataFrame(X_scaled, columns=data.feature_names).corr()
sns.heatmap(corr, cmap='RdBu_r', center=0, ax=ax, xticklabels=True, yticklabels=True)
ax.set_title('Feature Correlation Matrix (30 features)', fontsize=14)
plt.tight_layout(); plt.show()
print(f'Highly correlated pairs (|r| > 0.9): {(np.abs(corr.values[np.triu_indices(30, k=1)]) > 0.9).sum()}')

## 14. Univariate Analysis
### Explained variance by component

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(explained)+1), explained, color='steelblue')
axes[0].set_title('Explained Variance per Component')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Ratio')

axes[1].plot(range(1, len(cumulative)+1), cumulative, 'o-', color='coral')
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% threshold')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance')
axes[1].legend()

plt.tight_layout(); plt.show()

n_95 = np.argmax(cumulative >= 0.95) + 1
print(f'Components for 95% variance: {n_95}')
print(f'PC1 explains: {explained[0]:.1%}')
print(f'PC1+PC2 explain: {cumulative[1]:.1%}')

## 15. Bivariate / Multivariate Analysis
### 2D PCA Projection

In [ ]:
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca2.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
for label, name, color in [(0, 'Malignant', '#e63946'), (1, 'Benign', '#457b9d')]:
    mask = y == label
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], label=name, alpha=0.6, color=color, s=40)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('2D PCA Projection — Breast Cancer Dataset')
ax.legend()
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Component Loadings — What Drives Each PC?

In [ ]:
loadings = pd.DataFrame(pca2.components_.T, columns=['PC1','PC2'], index=data.feature_names)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for i, pc in enumerate(['PC1','PC2']):
    sorted_loads = loadings[pc].sort_values()
    sorted_loads.plot(kind='barh', ax=axes[i], color=sorted_loads.apply(lambda x: 'steelblue' if x > 0 else 'coral'))
    axes[i].set_title(f'{pc} Loadings')
    axes[i].axvline(0, color='black', linewidth=0.5)
plt.tight_layout(); plt.show()

print('Top 5 features for PC1:')
print(loadings['PC1'].abs().nlargest(5).to_string())

## 17. Statistical Checks / Hypothesis Testing
**Test:** Are the PCA-projected means significantly different between malignant and benign?

In [ ]:
from scipy import stats
for i, pc_name in enumerate(['PC1','PC2']):
    mal = X_2d[y==0, i]
    ben = X_2d[y==1, i]
    t, p = stats.ttest_ind(mal, ben)
    print(f'{pc_name}: malignant mean={mal.mean():.2f}, benign mean={ben.mean():.2f}, t={t:.2f}, p={p:.2e}')

## 18. Visual Analysis
### PCA Without Scaling vs With Scaling — Why It Matters

In [ ]:
pca_noscale = PCA(n_components=2, random_state=RANDOM_STATE)
X_noscale = pca_noscale.fit_transform(X)  # NOT scaled

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for label, name, color in [(0,'Malignant','#e63946'),(1,'Benign','#457b9d')]:
    mask = y == label
    axes[0].scatter(X_noscale[mask,0], X_noscale[mask,1], label=name, alpha=0.5, color=color)
    axes[1].scatter(X_2d[mask,0], X_2d[mask,1], label=name, alpha=0.5, color=color)

axes[0].set_title('PCA WITHOUT Scaling')
axes[0].legend()
axes[1].set_title('PCA WITH Scaling (correct)')
axes[1].legend()
plt.tight_layout(); plt.show()

print(f'Without scaling — PC1 variance: {pca_noscale.explained_variance_ratio_[0]:.1%}')
print(f'With scaling    — PC1 variance: {pca2.explained_variance_ratio_[0]:.1%}')

## 19. Key Findings
1. **2 components explain ~63% of variance** — sufficient for visualisation but not for modelling.
2. **10 components capture 95%** of total variance (from 30 original features).
3. **Malignant and benign clusters are well-separated** in the 2D PCA space.
4. **Scaling is critical** — without it, PCA is dominated by features with the largest absolute values.
5. **PC1 loadings** are dominated by size-related features (mean area, mean perimeter).

## 20. Limitations
- PCA is a linear method — non-linear relationships are not captured.
- Interpretability decreases with PCA — components are abstract combinations.
- Explained variance percentage depends on scaling.
- PCA does not consider class labels (unsupervised).

## 21. Recommendations / Next Steps
1. Apply t-SNE or UMAP for non-linear dimensionality reduction comparison.
2. Use PCA-reduced features as input to classifiers (logistic regression, SVM).
3. Experiment with kernel PCA for non-linear projections.
4. Compare PCA with LDA (which uses class labels).
5. Investigate the biological meaning of top-loading features in each component.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Applying PCA without scaling | Large-scale features dominate | Always use StandardScaler first |
| Choosing components by count, not variance | Arbitrary and may lose signal | Use cumulative variance threshold (e.g., 95%) |
| Interpreting PCs as original features | PCs are linear combinations | Examine loadings to understand composition |
| Using PCA for feature selection | PCA creates new features, not selects | Use feature importance or L1 regularisation |
| Forgetting to transform test data with the same PCA | Data leakage or mismatch | Fit on train, transform both |

## 23. Mini Challenge / Exercises
1. **3D PCA:** Add a third component and create a 3D scatter plot. Does it improve separation?
2. **Reconstruction error:** Project data to 2D then reconstruct — what is the mean squared error?
3. **Biplot:** Overlay feature vectors on the 2D scatter to create a biplot.
4. **Kernel PCA:** Apply `KernelPCA` with RBF kernel — does it improve class separation?
5. **Incremental PCA:** Use `IncrementalPCA` for the same data — do results differ?

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  PCA reduces 30 features to 2 with ~63% variance retained — enough for visualisation.
TAKEAWAY 2  StandardScaler before PCA is non-negotiable for meaningful results.
TAKEAWAY 3  Malignant and benign tumours form visually separable clusters in PCA space.
TAKEAWAY 4  PC1 is driven by tumour size features; PC2 by texture and smoothness.
TAKEAWAY 5  10 components capture 95% of variance — a 3× dimensionality reduction.
```